# EJERCICIO FINAL PYSPARK (DOCKER + KAFKA + PYSPARK)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window


## Creamos la sesión de spark

In [2]:
spark = (
    SparkSession.builder
        .appName("ProyectoIoT")
        .getOrCreate()
)
spark

spark.readStream
Indica que vamos a leer datos en modo streaming, es decir, datos que llegan continuamente.

.format("kafka")
Le decimos a Spark que la fuente de datos de streaming será Kafka.

.option("kafka.bootstrap.servers", "kafka:9092")
Especifica la dirección del clúster Kafka al que conectarse.
En este caso "kafka" es el nombre del contenedor Docker y 9092 es el puerto del broker.

.option("subscribe", "sensores")
Indica el topic de Kafka del que queremos leer mensajes → "sensores".

.option("startingOffsets", "latest")
Define desde qué punto empezar a leer:

"latest" → solo recibe mensajes nuevos, enviados después de iniciar el streaming.

"earliest" sería para leer todos los mensajes antiguos también.

.load()
Ejecuta la configuración y crea un DataFrame de streaming, donde cada fila representa un mensaje recibido desde Kafka.

In [3]:
raw_df = (
    spark.readStream
         .format("kafka")
         .option("kafka.bootstrap.servers", "kafka:9092")  # <-- nombre del contenedor
         .option("subscribe", "sensores")
         .option("startingOffsets", "latest")
         .load()
)


Este bloque transforma los mensajes de Kafka en un formato que Spark puede procesar:

Define la estructura de los datos que esperamos recibir (sensor, valor, temperatura, humedad, estado, timestamp y UUID).

Convierte el JSON recibido en columnas individuales para poder trabajar con cada campo.

Crea una columna de tiempo (event_time) a partir del timestamp original para poder usarlo en ventanas y agregaciones temporales.

In [4]:
schema = StructType([
    StructField("sensor_id", StringType()),
    StructField("value", DoubleType()),
    StructField("temperature", DoubleType()),
    StructField("humidity", DoubleType()),
    StructField("status", StringType()),
    StructField("timestamp", DoubleType()),
    StructField("uuid", StringType())
])

df = raw_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

df = df.withColumn("event_time", col("timestamp").cast("timestamp"))


Este bloque realiza agregaciones por ventana de tiempo sobre el DataFrame de streaming:

Define un watermark de 1 minuto para indicar a Spark hasta qué punto los datos antiguos se pueden considerar válidos. Esto ayuda a manejar retrasos y datos tardíos.

Agrupa los datos cada 30 segundos por sensor (sensor_id).

Calcula estadísticas dentro de cada ventana: promedio de valor, temperatura y humedad, y cuenta el número de eventos recibidos.

El resultado es un DataFrame de streaming con resúmenes temporales por sensor listo para análisis o escritura.

In [5]:
agg_df = (
    df
    .withWatermark("event_time", "1 minute")   # ← IMPORTANTE
    .groupBy(
        window(col("event_time"), "30 seconds"),
        col("sensor_id")
    )
    .agg(
        avg("value").alias("avg_value"),
        avg("temperature").alias("avg_temp"),
        avg("humidity").alias("avg_humidity"),
        count("*").alias("num_events")
    )
)


Este bloque escribe los resultados del streaming en archivos Parquet de manera continua:

Define la carpeta de salida donde se guardarán los archivos Parquet (resultados/).

Usa un checkpoint (chk/) para que Spark recuerde qué datos ya procesó y pueda reiniciar de forma segura en caso de fallo.

Modo append: los nuevos datos se agregan continuamente a los archivos existentes.

Inicia el streaming, haciendo que las agregaciones se escriban en tiempo real mientras llegan nuevos datos.

El resultado es un conjunto de archivos Parquet que refleja las métricas agregadas por ventana y por sensor.

In [6]:
parquet_query = (
    agg_df
    .writeStream
    .format("parquet")
    .option("path", "resultados/")
    .option("checkpointLocation", "chk/")
    .outputMode("append")
    .start()
)


A partir de aquí, te toca a ti, sigue las cuestiones planteadas en el Readme y completa el notebook. Despues guardatelo con los outputs de las celdas y súbelo al repo. Mucha suerte que ya lo teneis ;)

### Ejercicio 1

In [18]:
df_final = spark.read.parquet("resultados")
df_final.printSchema()
total_filas = df_final.count()
print(f"Número total de filas (ventanas procesadas): {total_filas}")
num_sensores = df_final.select("sensor_id").distinct().count()
print(f"Número de sensores distintos detectados: {num_sensores}")

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- avg_value: double (nullable = true)
 |-- avg_temp: double (nullable = true)
 |-- avg_humidity: double (nullable = true)
 |-- num_events: long (nullable = false)

Número total de filas (ventanas procesadas): 142
Número de sensores distintos detectados: 4


### Ejercicio 2

In [20]:
from pyspark.sql.functions import col, min, max, avg

df_relacion = df_final.withColumn(
    "delta_clima",
    col("avg_temp") - col("avg_humidity")
)

resultado = df_relacion.select(
    min("delta_clima").alias("Valor_Minimo"),
    max("delta_clima").alias("Valor_Maximo"),
    avg("delta_clima").alias("Valor_Medio")
)

resultado.show()

df_relacion.select(
    "sensor_id",
    "avg_temp",
    "avg_humidity",
    "delta_clima"
).show(5)


+-------------------+------------------+-------------------+
|       Valor_Minimo|      Valor_Maximo|        Valor_Medio|
+-------------------+------------------+-------------------+
|-37.940000000000005|42.010000000000005|0.05916034415506223|
+-------------------+------------------+-------------------+

+---------+------------------+------------------+-------------------+
|sensor_id|          avg_temp|      avg_humidity|        delta_clima|
+---------+------------------+------------------+-------------------+
|       S1|             30.64|             63.94|              -33.3|
|       S1|45.526666666666664|59.748333333333335|-14.221666666666671|
|       S3| 55.63199999999999|48.291000000000004|  7.340999999999987|
|       S3| 49.04111111111111| 49.74444444444444|-0.7033333333333331|
|       S4|             76.29| 63.59666666666667| 12.693333333333335|
+---------+------------------+------------------+-------------------+
only showing top 5 rows



### Ejercicio 3

In [36]:
from pyspark.sql.functions import col

# Aplicar filtro con varias condiciones
df_filtered = df_final.filter(
    (col("avg_humidity") > 50) &
    (col("num_events") > 0) &
    (col("sensor_id") == "S2")
)

# Contar registros
cantidad = df_filtered.count()

print("Condiciones aplicadas: Humedad > 50 AND Eventos > 0 AND Sensor == S2")
print(f"Total de registros que cumplen todo a la vez: {cantidad}")

print("--- Muestra de los registros filtrados ---")
df_filtered.show(5)

Condiciones aplicadas: Humedad > 50 AND Eventos > 0 AND Sensor == S2
Total de registros que cumplen todo a la vez: 27
--- Muestra de los registros filtrados ---
+--------------------+---------+------------------+------------------+-----------------+----------+
|              window|sensor_id|         avg_value|          avg_temp|     avg_humidity|num_events|
+--------------------+---------+------------------+------------------+-----------------+----------+
|{2026-02-08 17:31...|       S2|27.196250000000003|48.088750000000005|          54.4675|         8|
|{2026-02-08 17:32...|       S2|28.582307692307687| 48.65538461538461|60.06000000000001|        13|
|{2026-02-08 17:33...|       S2|27.606666666666666| 58.79222222222222|58.78777777777777|         9|
|{2026-02-08 17:34...|       S2|             33.79| 58.08444444444444|59.62777777777777|         9|
|{2026-02-08 17:34...|       S2|28.827999999999996|            61.178|57.99400000000001|         5|
+--------------------+---------+-------

### Ejercicio 4

In [37]:
from pyspark.sql.functions import avg, max, count

df_agregado = df_final.groupBy("sensor_id").agg(
    avg("avg_value").alias("media_avg_value"),
    max("avg_temp").alias("temp_maxima"),
    count("*").alias("num_ventanas")
)

df_agregado.show()
from pyspark.sql.functions import desc

df_agregado.orderBy(desc("media_avg_value")).show(1)
df_agregado.orderBy(desc("temp_maxima")).show(1)
df_agregado.select("sensor_id", "num_ventanas").show()


+---------+------------------+-----------+------------+
|sensor_id|   media_avg_value|temp_maxima|num_ventanas|
+---------+------------------+-----------+------------+
|       S4| 29.27847380952381|      76.29|          35|
|       S3|29.002612243466412|      74.45|          36|
|       S1|28.713926702926706|     70.662|          36|
|       S2| 29.63899745334031|     72.172|          35|
+---------+------------------+-----------+------------+

+---------+-----------------+-----------+------------+
|sensor_id|  media_avg_value|temp_maxima|num_ventanas|
+---------+-----------------+-----------+------------+
|       S2|29.63899745334031|     72.172|          35|
+---------+-----------------+-----------+------------+
only showing top 1 row

+---------+-----------------+-----------+------------+
|sensor_id|  media_avg_value|temp_maxima|num_ventanas|
+---------+-----------------+-----------+------------+
|       S4|29.27847380952381|      76.29|          35|
+---------+-----------------+---

### Ejercicio 5

In [38]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, desc

ventana_sensor = Window.partitionBy("sensor_id").orderBy(desc("avg_value"))

df_rank = df_final.withColumn(
    "rank",
    row_number().over(ventana_sensor)
).filter(col("rank") == 1)
df_rank.orderBy(desc("avg_value")).show(1)


+--------------------+---------+------------------+--------+------------+----------+----+
|              window|sensor_id|         avg_value|avg_temp|avg_humidity|num_events|rank|
+--------------------+---------+------------------+--------+------------+----------+----+
|{2026-02-08 17:40...|       S2|42.480000000000004|  50.495|     58.2125|         4|   1|
+--------------------+---------+------------------+--------+------------+----------+----+
only showing top 1 row



### Ejercicio 6

In [39]:
from pyspark.sql import Row

df_sensores_info = spark.createDataFrame([
    Row(sensor_id="S1", location="Valencia", tipo="temperatura"),
    Row(sensor_id="S2", location="Madrid",  tipo="humedad"),
    Row(sensor_id="S3", location="Barcelona", tipo="mixto")
])


df_join = df_final.join(
    df_sensores_info,
    on="sensor_id",
    how="left"
)
df_join.show(5, truncate=False)


+---------+------------------------------------------+------------------+------------------+------------------+----------+---------+-----------+
|sensor_id|window                                    |avg_value         |avg_temp          |avg_humidity      |num_events|location |tipo       |
+---------+------------------------------------------+------------------+------------------+------------------+----------+---------+-----------+
|S4       |{2026-02-08 17:32:00, 2026-02-08 17:32:30}|28.536666666666665|76.29             |63.59666666666667 |3         |NULL     |NULL       |
|S3       |{2026-02-08 17:31:30, 2026-02-08 17:32:00}|29.8              |55.63199999999999 |48.291000000000004|10        |Barcelona|mixto      |
|S3       |{2026-02-08 17:32:00, 2026-02-08 17:32:30}|32.70333333333333 |49.04111111111111 |49.74444444444444 |9         |Barcelona|mixto      |
|S1       |{2026-02-08 17:31:00, 2026-02-08 17:31:30}|21.29             |30.64             |63.94             |1         |Valencia

### Ejercicio 7

In [40]:
from pyspark.sql.functions import col, date_trunc, count, avg

df_time = df_final.withColumn(
    "minute",
    date_trunc("minute", col("window.start"))
)
df_time_agg = df_time.groupBy("minute").agg(
    count("*").alias("num_ventanas"),
    avg("avg_humidity").alias("humedad_media")
)


### Ejercicio 8

In [41]:
from pyspark.sql.functions import spark_partition_id

df_particionado = df_final.repartition("sensor_id") \
    .withColumn("partition_id", spark_partition_id())
df_particionado.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()


+------------+-----+
|partition_id|count|
+------------+-----+
|           0|  142|
+------------+-----+



### Ejercicio 9

In [42]:
from pyspark.sql.functions import avg, stddev

stats = df_final.select(
    avg("avg_value").alias("mean_value"),
    stddev("avg_value").alias("std_value"),
    avg("avg_temp").alias("mean_temp"),
    stddev("avg_temp").alias("std_temp")
).collect()[0]
from pyspark.sql.functions import col

df_anom = df_final.withColumn(
    "anomaly",
    (col("avg_value") > stats["mean_value"] + stats["std_value"]) |
    (col("avg_temp")  > stats["mean_temp"]  + stats["std_temp"])
)
df_anom.filter(col("anomaly")).count()
from pyspark.sql.functions import desc

df_anom.filter(col("anomaly")) \
    .groupBy("sensor_id") \
    .count() \
    .orderBy(desc("count")) \
    .show(1)


+---------+-----+
|sensor_id|count|
+---------+-----+
|       S4|   11|
+---------+-----+
only showing top 1 row



### Ejercicio 10

In [43]:
from pyspark.sql.functions import col

df_score = df_final.withColumn(
    "score",
    col("avg_value") * 0.5 +
    col("avg_temp") * 0.3 +
    col("avg_humidity") * 0.2
)
from pyspark.sql.functions import avg, desc

df_score_sensor = df_score.groupBy("sensor_id").agg(
    avg("score").alias("score_media")
)
df_score_sensor.orderBy(desc("score_media")).show(1)


+---------+-----------------+
|sensor_id|      score_media|
+---------+-----------------+
|       S3|42.41150271364438|
+---------+-----------------+
only showing top 1 row



### Ejercicio 11